In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../../data/preprocessed/X_binned.csv')

feature_cols = [col for col in df.columns if col not in
                ["result_match", "stage", "season", "date", "home_team", "away_team", "match_api_id"]]

y = df[['result_match']]
X = df[feature_cols].copy()

In [3]:
from xgboost import XGBClassifier

params = {'colsample_bytree': 0.8949861281692281,
          'gamma': 3.5920440761581354,
          'learning_rate': 0.2861004914177646,
          'max_depth': 12,
          'min_child_weight': 13.500372304245893,
          'n_estimators': 350,
          'reg_alpha': 0.1842741694829661,
          'reg_lambda': 0.008596007166608277,
          'subsample': 0.8552978617018039}

y_train_value_counts = y.value_counts()
ratio = y_train_value_counts[0] / y_train_value_counts[1]

model = XGBClassifier(
    **params,
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=ratio
)


In [12]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import make_scorer, f1_score

tscv = TimeSeriesSplit(n_splits=4)

f1 = make_scorer(f1_score)
selector = SequentialFeatureSelector(estimator=model,
                                     n_features_to_select="auto",
                                     tol=0.0001,
                                     direction="forward",
                                     scoring=f1,
                                     cv=tscv)
selector.fit(X, y)
selected_features = selector.support_
selected_data = X.columns[selected_features]
print(selected_data)

Index(['points_diff_binned'], dtype='object')
